# Ames Mutagenicity: Classical and Graph Neural Network Models

## Part 5 — Robustness checks and interval estimates

This notebook completes the reporting for the analyses carried out in Parts 2 to 4. It
covers the checks that establish whether a reported conclusion depends on a threshold, a
binning, or a single point estimate, and it repeats the data-randomisation control under
conditions that allow the permuted labels to be fitted.

**Contents**

1. Environment and shared definitions
2. Data and representative models
3. Sensitivity of toxicophore agreement to the retained fraction
4. Fingerprint bit collisions
5. Significance of the attribution rankings
6. Interval estimates for accuracy and calibration
7. Conformal prediction: marginal and class-conditional
8. Descriptor and linear baselines
9. Attribution analysis on further partitions
10. Per-seed variation
11. Data randomisation with regularisation disabled

*Approximate runtime: 4-5 hours.*

## 1. Environment and shared definitions

In [1]:
!pip install -q rdkit
!pip install -q xgboost
!pip install -q shap
!pip install -q torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 33.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 83.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK    = "/content/drive/MyDrive/ames_pipeline"
DATADIR, OUTDIR = f"{WORK}/data", f"{WORK}/results"
MODELS, EXPL    = f"{WORK}/models", f"{WORK}/explain"
for d in (WORK, DATADIR, OUTDIR, MODELS, EXPL):
    os.makedirs(d, exist_ok=True)
print("Working directory:", WORK)

Mounted at /content/drive
Working directory: /content/drive/MyDrive/ames_pipeline


In [3]:
%%writefile /content/ames_lib.py
"""Shared definitions for the Ames mutagenicity pipeline."""

import random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import rdFingerprintGenerator

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (GCNConv, GINConv, GINEConv, GATConv,
                                global_mean_pool, global_add_pool)
from sklearn.metrics import roc_auc_score

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")

FP_RADIUS, FP_BITS = 2, 2048
ATOM_DIM, BOND_DIM = 24, 6
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ATOM_LIST     = ['C','N','O','S','F','Cl','Br','I','P','B','Si','Se','other']
HYBRIDIZATION = ['SP','SP2','SP3','SP3D','SP3D2','other']
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def safe_scaffold(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    try:
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        try:
            return Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))
        except Exception:
            return smi


def make_scaffold_split(d, seed, frac=(0.8, 0.1, 0.1)):
    groups = defaultdict(list)
    for i, smi in zip(d.index, d["canonical"].values):
        groups[safe_scaffold(smi)].append(i)
    gs  = list(groups.values())
    rng = np.random.default_rng(seed)
    gs.sort(key=lambda g: (-len(g), rng.random()))
    n = len(d)
    n_train, n_train_valid = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    tr, va, te = [], [], []
    for g in gs:
        if len(tr) + len(g) <= n_train:
            tr += g
        elif len(tr) + len(va) + len(g) <= n_train_valid:
            va += g
        else:
            te += g
    return d.loc[tr], d.loc[va], d.loc[te]


def fp_array(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(morgan_gen.GetFingerprint(mol), arr)
    return arr


def build_X(d):
    X, keep = [], []
    for i, row in d.iterrows():
        a = fp_array(row["canonical"])
        if a is not None:
            X.append(a); keep.append(i)
    return np.array(X), d.loc[keep].reset_index(drop=True)


def one_hot(x, allowable):
    if x not in allowable:
        x = allowable[-1]
    return [int(x == a) for a in allowable]


def atom_features(atom):
    f  = one_hot(atom.GetSymbol(), ATOM_LIST)
    f += one_hot(str(atom.GetHybridization()), HYBRIDIZATION)
    f += [atom.GetDegree(), atom.GetFormalCharge(), int(atom.GetIsAromatic()),
          atom.GetTotalNumHs(), int(atom.IsInRing())]
    return f


def bond_features(bond):
    bt = bond.GetBondType()
    return [int(bt == Chem.BondType.SINGLE), int(bt == Chem.BondType.DOUBLE),
            int(bt == Chem.BondType.TRIPLE), int(bt == Chem.BondType.AROMATIC),
            int(bond.GetIsConjugated()), int(bond.IsInRing())]


def smiles_to_data(smiles, label):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() == 0:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    ei, ea = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        ei += [[i, j], [j, i]]
        ea += [bf, bf]
    if len(ei) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, BOND_DIM), dtype=torch.float)
    else:
        edge_index = torch.tensor(ei, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(ea, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr,
                y=torch.tensor([label], dtype=torch.float), smiles=smiles)


def build_dataset(d):
    graphs = [smiles_to_data(r["canonical"], float(r["label"])) for _, r in d.iterrows()]
    return [g for g in graphs if g is not None]


class GNN(nn.Module):
    def __init__(self, kind, in_dim=ATOM_DIM, edge_dim=BOND_DIM, hidden=128,
                 n_layers=3, dropout=0.2, heads=4, residual=True, use_edge_attr=True):
        super().__init__()
        self.kind, self.dropout, self.residual = kind, dropout, residual
        self.use_edge_attr = use_edge_attr and kind in ("gin", "gat")
        self.lin_in = nn.Linear(in_dim, hidden)
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layers):
            if kind == "gcn":
                conv = GCNConv(hidden, hidden)
            elif kind == "gin":
                mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden))
                conv = (GINEConv(mlp, train_eps=True, edge_dim=edge_dim)
                        if self.use_edge_attr else GINConv(mlp, train_eps=True))
            elif kind == "gat":
                conv = GATConv(hidden, hidden // heads, heads=heads,
                               edge_dim=edge_dim if self.use_edge_attr else None)
            else:
                raise ValueError(f"Unknown architecture: {kind}")
            self.convs.append(conv)
            self.bns.append(nn.BatchNorm1d(hidden))
        self.head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hidden, 1))

    def forward(self, data):
        x, edge_index, edge_attr, batch = (data.x, data.edge_index,
                                           data.edge_attr, data.batch)
        x = self.lin_in(x)
        for conv, bn in zip(self.convs, self.bns):
            h = (conv(x, edge_index, edge_attr) if self.use_edge_attr
                 else conv(x, edge_index))
            h = F.relu(bn(h))
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h if self.residual else h
        g = torch.cat([global_mean_pool(x, batch), global_add_pool(x, batch)], dim=1)
        return self.head(g).squeeze(-1)


def build_gnn(kind, hidden, n_layers, dropout, residual=True, heads=4,
              use_edge_attr=True):
    return GNN(kind, hidden=hidden, n_layers=n_layers, dropout=dropout,
               residual=residual, heads=heads,
               use_edge_attr=use_edge_attr).to(DEVICE)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    ys, ps, logits = [], [], []
    for d in loader:
        d = d.to(DEVICE)
        logit = model(d)
        ys.append(d.y.view(-1).cpu().numpy())
        ps.append(torch.sigmoid(logit).cpu().numpy())
        logits.append(logit.cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(logits)


class TensorView:
    def __init__(self, x, edge_index, edge_attr, batch):
        self.x, self.edge_index, self.edge_attr, self.batch = (
            x, edge_index, edge_attr, batch)


def train_gnn(kind, params, seed, tr_set, va_set, pos_weight,
              max_epochs=250, patience=30, use_edge_attr=True):
    """Train one graph model, retaining the epoch with the best validation ROC-AUC."""
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], params["dropout"],
                      params.get("residual", True), use_edge_attr=use_edge_attr)
    optimiser = torch.optim.Adam(model.parameters(), lr=params["lr"],
                                 weight_decay=params["weight_decay"])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    train_loader = DataLoader(tr_set, batch_size=params["batch_size"],
                              shuffle=True, drop_last=True)
    valid_loader = DataLoader(va_set, batch_size=256)
    best_auc, best_state, waited = -1.0, None, 0
    for epoch in range(max_epochs):
        model.train()
        for d in train_loader:
            d = d.to(DEVICE)
            optimiser.zero_grad()
            criterion(model(d), d.y.view(-1)).backward()
            optimiser.step()
        y_va, p_va, _ = evaluate(model, valid_loader)
        auc = roc_auc_score(y_va, p_va) if len(np.unique(y_va)) > 1 else 0.5
        if auc > best_auc:
            best_auc, waited = auc, 0
            best_state = {k: v.detach().cpu().clone()
                          for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_auc


def train_to_fit(kind, params, seed, dataset, n_epochs, dropout=0.0,
                 weight_decay=0.0, use_edge_attr=True):
    """Fit a dataset without regularisation or early stopping.

    Dropout, weight decay and class weighting are all disabled: the objective is
    to fit the supplied targets, not to generalise from them. The final partial
    batch is dropped because batch normalisation statistics computed on a very
    small batch can drive the network to a constant output, which leaves no
    usable gradient for attribution.
    """
    set_seed(seed)
    model = build_gnn(kind, params["hidden"], params["n_layers"], dropout=dropout,
                      residual=params.get("residual", True), use_edge_attr=use_edge_attr)
    optimiser = torch.optim.Adam(model.parameters(), lr=params["lr"],
                                 weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(dataset, batch_size=params["batch_size"],
                        shuffle=True, drop_last=True)
    last = float("nan")
    for epoch in range(n_epochs):
        model.train(); total = 0.0
        for d in loader:
            d = d.to(DEVICE)
            optimiser.zero_grad()
            loss = criterion(model(d), d.y.view(-1))
            loss.backward(); optimiser.step()
            total += loss.item()
        last = total / max(len(loader), 1)
        if (epoch + 1) % 100 == 0:
            print(f"    epoch {epoch+1:4d}  training loss = {last:.4f}")
    model.eval()
    return model, last

Writing /content/ames_lib.py


In [4]:
import sys, json, time
sys.path.insert(0, "/content")
import numpy as np, pandas as pd, torch
from ames_lib import *
from torch_geometric.loader import DataLoader
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score,
                             average_precision_score, matthews_corrcoef,
                             recall_score, accuracy_score)
from scipy import stats
from scipy.stats import wilcoxon

SEED, PRIMARY_SPLIT_SEED, EXPLAIN_SEED = 42, 101, 42
SEEDS, K_FRAC, N_IG_STEPS = [42, 43, 44], 0.30, 32
N_BOOTSTRAP  = 10000
N_NULL_SWEEP = 100      # untrained draws used for the retained-fraction sweep
K_GRID = [0.10, 0.20, 0.30, 0.40, 0.50]

HP_GNN     = json.load(open(f"{OUTDIR}/hpo_gnn.json"))
HP_CLASSIC = json.load(open(f"{OUTDIR}/hpo_classic.json"))
print("Device:", DEVICE)

Device: cuda


## 2. Data and representative models

In [5]:
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

df = pd.concat([pd.read_csv(f"{DATADIR}/{f}")
                for f in ["train.csv", "valid.csv", "test.csv"]],
               ignore_index=True)[["Drug_ID", "smiles", "label"]]
df["canonical"] = df["smiles"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None)
df = df.dropna(subset=["canonical"]).drop_duplicates(
    subset="canonical", keep="first").reset_index(drop=True)

df_train, df_valid, df_test = [x.reset_index(drop=True)
    for x in make_scaffold_split(df, seed=PRIMARY_SPLIT_SEED)]
X_train, df_train = build_X(df_train); y_train = df_train["label"].values
X_valid, df_valid = build_X(df_valid); y_valid = df_valid["label"].values
X_test,  df_test  = build_X(df_test);  y_test  = df_test["label"].values
train_set, valid_set, test_set = (build_dataset(df_train), build_dataset(df_valid),
                                  build_dataset(df_test))
valid_loader = DataLoader(valid_set, batch_size=256)
test_loader  = DataLoader(test_set,  batch_size=256)

# Scaffold leakage, recorded as an artefact.
sc_tr = {safe_scaffold(s) for s in df_train["canonical"]}
sc_te = {safe_scaffold(s) for s in df_test["canonical"]}
sc_va = {safe_scaffold(s) for s in df_valid["canonical"]}
leak = {"test_scaffolds": len(sc_te),
        "shared_with_train": len(sc_te & sc_tr),
        "pct_shared_test_train": 100 * len(sc_te & sc_tr) / max(len(sc_te), 1),
        "pct_shared_valid_train": 100 * len(sc_va & sc_tr) / max(len(sc_va), 1)}
pd.DataFrame([leak]).to_csv(f"{OUTDIR}/scaffold_leakage.csv", index=False)
print(leak)

hp = HP_GNN["gin"]
gin_rep = build_gnn("gin", hp["hidden"], hp["n_layers"], hp["dropout"],
                    hp.get("residual", True))
gin_rep.load_state_dict(torch.load(f"{MODELS}/gin_seed{EXPLAIN_SEED}.pt",
                                   map_location=DEVICE)["state_dict"])
gin_rep.eval()

spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
xgb_rep = XGBClassifier(n_estimators=1500, **HP_CLASSIC["XGBoost"],
                        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
                        eval_metric="auc", early_stopping_rounds=50, tree_method="hist",
                        random_state=EXPLAIN_SEED, n_jobs=-1)
xgb_rep.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
print("Representative models ready")

{'test_scaffolds': 726, 'shared_with_train': 0, 'pct_shared_test_train': 0.0, 'pct_shared_valid_train': 0.0}
Representative models ready


In [6]:
TOXICOPHORES = {
    "aromatic_nitro":         "[a][N+](=O)[O-]",
    "aromatic_amine":         "[a][NX3;H2,H1;!$(N[!#6]);!$(N=*)]",
    "aromatic_nitroso":       "[a][NX2]=[OX1]",
    "alkyl_nitrite":          "[CX4][OX2][NX2]=[OX1]",
    "nitrosamine":            "[NX3][NX2]=[OX1]",
    "epoxide":                "C1OC1",
    "aziridine":              "C1NC1",
    "azide":                  "[NX2]=[NX2+]=[NX1-]",
    "azo_aromatic":           "[a][NX2]=[NX2][a]",
    "aliphatic_halide":       "[CX4][F,Cl,Br,I]",
    "hetero_hetero_bond":     "[N,O;H1,H2][N,O;H1,H2]",
    "polycyclic_aromatic":    "c1ccc2ccccc2c1",
    "aromatic_hydroxylamine": "[a][NX3;H1][OX2;H1]",
    "michael_acceptor":       "[CX3]=[CX3][CX3]=[OX1]",
    "aromatic_azoxy":         "[a][NX2]=[NX3+]([O-])[a]",
    "nitrogen_mustard":       "[NX3]([CX4][CX4][F,Cl,Br,I])[CX4][CX4][F,Cl,Br,I]",
    "aromatic_hydrazine":     "[a][NX3;H1][NX3;H2]",
    "alkyl_sulfonate":        "[CX4][OX2][SX4](=[OX1])(=[OX1])",
}
TOX_PATTERNS = {k: Chem.MolFromSmarts(v) for k, v in TOXICOPHORES.items()}

def toxicophore_atoms(mol):
    atoms = set()
    for patt in TOX_PATTERNS.values():
        for m in mol.GetSubstructMatches(patt):
            atoms.update(m)
    return atoms

EVAL = []
for pos, (smi, lab) in enumerate(zip(df_test["canonical"], y_test)):
    if lab != 1:
        continue
    mol = Chem.MolFromSmiles(smi)
    if mol is None or mol.GetNumAtoms() == 0:
        continue
    tox = toxicophore_atoms(mol)
    if not tox or len(tox) == mol.GetNumAtoms():
        continue
    EVAL.append({"row": pos, "smiles": smi, "mol": mol, "tox": tox,
                 "n_atoms": mol.GetNumAtoms(), "chance": len(tox) / mol.GetNumAtoms()})
EVAL_GRAPHS = [smiles_to_data(e["smiles"], 1.0) for e in EVAL]
CHANCE, N_EVAL = np.array([e["chance"] for e in EVAL]), len(EVAL)

def overlap_from_scores(scores_list, k_frac=K_FRAC, tie_seed=0):
    rng = np.random.default_rng(tie_seed)
    out = np.full(N_EVAL, np.nan)
    for i, (e, sc) in enumerate(zip(EVAL, scores_list)):
        if sc is None or len(sc) != e["n_atoms"]:
            continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac * n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[i] = len(set(order[:k].tolist()) & e["tox"]) / k
    return out

def lift_from_overlap(ov):
    m = ~np.isnan(ov)
    return float(ov[m].mean() / CHANCE[m].mean()) if m.sum() else np.nan

def ceiling_at(k_frac):
    oracle = [np.array([1.0 if a in e["tox"] else 0.0 for a in range(e["n_atoms"])])
              for e in EVAL]
    return lift_from_overlap(overlap_from_scores(oracle, k_frac=k_frac, tie_seed=0))

print(f"Evaluation set: {N_EVAL} molecules")

Evaluation set: 237 molecules


## 3. Sensitivity of toxicophore agreement to the retained fraction

Agreement is reported at a retained fraction of 30% of atoms. The comparison is repeated
across a range of fractions, so that the ordering of the explainers and their position
relative to the untrained distribution can be seen not to depend on that choice. The
attainable ceiling changes with the fraction and is recomputed at each setting.

In [7]:
def batched_gradient_scores(model, graphs, method="ig", n_steps=N_IG_STEPS,
                            batch_size=64, signed=False):
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    out = []
    for batch in DataLoader(graphs, batch_size=batch_size, shuffle=False):
        batch = batch.to(DEVICE)
        x0 = batch.x.clone()
        if method == "ig":
            total = torch.zeros_like(x0)
            for alpha in torch.linspace(1.0 / n_steps, 1.0, n_steps):
                xa = (x0 * alpha).detach().requires_grad_(True)
                logit = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
                total += torch.autograd.grad(logit.sum(), xa)[0]
            attr = x0 * (total / n_steps)
        else:
            xa = x0.clone().detach().requires_grad_(True)
            logit = model(TensorView(xa, batch.edge_index, batch.edge_attr, batch.batch))
            grad = torch.autograd.grad(logit.sum(), xa)[0]
            attr = grad if method == "saliency" else x0 * grad
        scores = (attr.sum(dim=1) if signed else attr.abs().sum(dim=1))
        scores = scores.detach().cpu().numpy()
        b = batch.batch.cpu().numpy()
        for g in range(batch.num_graphs):
            out.append(scores[b == g])
    return out

import shap
from rdkit.Chem import rdFingerprintGenerator as rfg

def fp_with_bitinfo(mol):
    ao = rfg.AdditionalOutput(); ao.AllocateBitInfoMap()
    fp = morgan_gen.GetFingerprint(mol, additionalOutput=ao)
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr, ao.GetBitInfoMap()

def bit_to_atoms(mol, atom_idx, radius):
    if radius == 0:
        return {atom_idx}
    env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
    atoms = {atom_idx}
    for bidx in env:
        bond = mol.GetBondWithIdx(bidx)
        atoms.add(bond.GetBeginAtomIdx()); atoms.add(bond.GetEndAtomIdx())
    return atoms

BITINFO = [fp_with_bitinfo(e["mol"])[1] for e in EVAL]

def shap_to_atom_scores(shap_values, signed=True):
    sv = np.asarray(shap_values); per_mol = sv.ndim == 2
    out = []
    for i, (e, bi) in enumerate(zip(EVAL, BITINFO)):
        vec = sv[i] if per_mol else sv
        scores = np.zeros(e["n_atoms"], dtype=float)
        for bit, envs in bi.items():
            v = float(vec[bit])
            if v == 0.0 or len(envs) == 0:
                continue
            for atom_idx, radius in envs:
                atoms = bit_to_atoms(e["mol"], atom_idx, radius)
                share = v / (len(envs) * len(atoms))
                for a in atoms:
                    scores[a] += share
        out.append(scores if signed else np.abs(scores))
    return out

def tree_shap_matrix(model, X):
    v = shap.TreeExplainer(model).shap_values(X)
    if isinstance(v, list):
        v = v[-1]
    v = np.asarray(v)
    return v[:, :, -1] if v.ndim == 3 else v

X_eval = np.array([fp_array(e["smiles"]) for e in EVAL])
SCORES = {
    "XGBoost (TreeSHAP, signed)": shap_to_atom_scores(tree_shap_matrix(xgb_rep, X_eval)),
    "GIN (Integrated Gradients)": batched_gradient_scores(gin_rep, EVAL_GRAPHS, "ig"),
    "GIN (Gradient x Input)":     batched_gradient_scores(gin_rep, EVAL_GRAPHS, "gradinput"),
    "GIN (Saliency)":             batched_gradient_scores(gin_rep, EVAL_GRAPHS, "saliency"),
    "GIN (GNNExplainer)":         list(np.load(f"{EXPL}/gnnexplainer_scores.npy",
                                               allow_pickle=True)),
}

rows = []
for k in K_GRID:
    ceil_k = ceiling_at(k)
    for label, sc in SCORES.items():
        lift = lift_from_overlap(overlap_from_scores(sc, k_frac=k, tie_seed=SEED))
        rows.append({"k_fraction": k, "source": label, "lift": lift,
                     "ceiling": ceil_k,
                     "pct_of_ceiling": 100 * (lift - 1) / (ceil_k - 1)})

print(f"Untrained distribution at each retained fraction ({N_NULL_SWEEP} draws)")
t0 = time.time()
null_by_k = {k: [] for k in K_GRID}
for i in range(N_NULL_SWEEP):
    set_seed(10_000 + i)
    m = build_gnn("gin", hp["hidden"], hp["n_layers"], hp["dropout"],
                  hp.get("residual", True))
    sc = batched_gradient_scores(m, EVAL_GRAPHS, "ig")
    for k in K_GRID:
        null_by_k[k].append(
            lift_from_overlap(overlap_from_scores(sc, k_frac=k, tie_seed=10_000 + i)))
    del m; torch.cuda.empty_cache()
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{N_NULL_SWEEP}  ({(time.time()-t0)/60:.1f} min)")

for k in K_GRID:
    v = np.array(null_by_k[k]); ceil_k = ceiling_at(k)
    rows.append({"k_fraction": k, "source": f"untrained GIN (n={N_NULL_SWEEP})",
                 "lift": v.mean(), "ceiling": ceil_k,
                 "pct_of_ceiling": 100 * (v.mean() - 1) / (ceil_k - 1),
                 "null_p2_5": np.percentile(v, 2.5),
                 "null_p97_5": np.percentile(v, 97.5)})

topk_df = pd.DataFrame(rows)
topk_df.to_csv(f"{OUTDIR}/table17_topk_sensitivity.csv", index=False)
print()
print(topk_df.pivot_table(index="source", columns="k_fraction",
                          values="lift").round(4).to_string())

Untrained distribution at each retained fraction (100 draws)
  25/100  (0.3 min)
  50/100  (0.5 min)
  75/100  (0.8 min)
  100/100  (1.1 min)

k_fraction                     0.1     0.2     0.3     0.4     0.5
source                                                            
GIN (GNNExplainer)          0.8337  0.8581  0.8978  0.9557  0.9752
GIN (Gradient x Input)      1.2973  1.2407  1.2164  1.1606  1.1325
GIN (Integrated Gradients)  1.3619  1.2842  1.2265  1.1724  1.1304
GIN (Saliency)              1.1885  1.1911  1.1784  1.1858  1.1582
XGBoost (TreeSHAP, signed)  1.8015  1.6949  1.5422  1.4212  1.3183
untrained GIN (n=100)       1.1836  1.1440  1.1187  1.0890  1.0640


## 4. Fingerprint bit collisions

A folded fingerprint maps distinct atom environments onto shared bits. The proportion of
bits that carry more than one environment bounds how precisely a bit-level attribution can
be assigned to particular atoms.

In [8]:
from collections import defaultdict as dd
env_per_bit = dd(set)
n_mols = 0
for _, row in pd.concat([df_train, df_valid, df_test]).iterrows():
    mol = Chem.MolFromSmiles(row["canonical"])
    if mol is None:
        continue
    n_mols += 1
    _, bi = fp_with_bitinfo(mol)
    for bit, envs in bi.items():
        for atom_idx, radius in envs:
            env = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom_idx)
            if radius == 0:
                key = (mol.GetAtomWithIdx(atom_idx).GetSymbol(), 0)
            else:
                sub = Chem.PathToSubmol(mol, env)
                key = (Chem.MolToSmiles(sub), radius)
            env_per_bit[bit].add(key)

set_bits = len(env_per_bit)
colliding = [b for b, s in env_per_bit.items() if len(s) > 1]
report = {
    "n_molecules": n_mols, "fp_bits": FP_BITS, "n_set_bits": set_bits,
    "n_colliding_bits": len(colliding),
    "pct_colliding": 100 * len(colliding) / max(set_bits, 1),
    "mean_environments_per_colliding_bit":
        float(np.mean([len(env_per_bit[b]) for b in colliding])) if colliding else 0.0,
    "max_environments_on_one_bit": max(len(s) for s in env_per_bit.values()),
}
pd.DataFrame([report]).to_csv(f"{OUTDIR}/table18_bit_collisions.csv", index=False)
for k, v in report.items():
    print(f"  {k:38s}: {round(float(v), 4)}")

  n_molecules                           : 7255.0
  fp_bits                               : 2048.0
  n_set_bits                            : 2048.0
  n_colliding_bits                      : 2048.0
  pct_colliding                         : 100.0
  mean_environments_per_colliding_bit   : 11.9448
  max_environments_on_one_bit           : 33.0


## 5. Significance of the attribution rankings

The attribution AUROC is computed per molecule. A signed-rank test against the value
expected from an uninformative ranking establishes whether each explainer differs from that
reference, and the p-values are corrected for the number of explainers compared.

In [9]:
# Bootstrap intervals for the lift of each explainer, with multiplicity correction.
rng_l = np.random.default_rng(SEED)
rows = []
for label, sc in SCORES.items():
    ov = overlap_from_scores(sc, tie_seed=SEED)
    valid = ~np.isnan(ov)
    o, c = ov[valid], CHANCE[valid]
    n = len(o)
    boot = np.array([o[i].mean() / c[i].mean()
                     for i in rng_l.integers(0, n, (N_BOOTSTRAP, n))])
    d = o - c
    stat, p = wilcoxon(d)
    rows.append({"explainer": label, "n": n, "lift": o.mean() / c.mean(),
                 "lift_ci_low": np.percentile(boot, 2.5),
                 "lift_ci_high": np.percentile(boot, 97.5),
                 "wilcoxon_p_raw": p})
lift_sig = pd.DataFrame(rows)
p = lift_sig["wilcoxon_p_raw"].values
order = np.argsort(p); k = len(p); adj = np.empty(k); run = 0.0
for r, i in enumerate(order):
    run = max(run, (k - r) * p[i]); adj[i] = min(run, 1.0)
lift_sig["wilcoxon_p_holm"] = adj
lift_sig["differs_from_chance"] = lift_sig["wilcoxon_p_holm"] < 0.05
lift_sig.to_csv(f"{OUTDIR}/table19b_lift_significance.csv", index=False)
print(lift_sig.round(4).to_string(index=False))

                 explainer   n   lift  lift_ci_low  lift_ci_high  wilcoxon_p_raw  wilcoxon_p_holm  differs_from_chance
XGBoost (TreeSHAP, signed) 237 1.5422       1.4615        1.6333          0.0000           0.0000                 True
GIN (Integrated Gradients) 237 1.2265       1.1711        1.2842          0.0000           0.0000                 True
    GIN (Gradient x Input) 237 1.2164       1.1512        1.2837          0.0000           0.0000                 True
            GIN (Saliency) 237 1.1784       1.1047        1.2585          0.0000           0.0000                 True
        GIN (GNNExplainer) 237 0.8978       0.8448        0.9518          0.0001           0.0001                 True


In [10]:
def attribution_auroc_per_molecule(scores_list):
    out = []
    for e, sc in zip(EVAL, scores_list):
        if sc is None or len(sc) != e["n_atoms"]:
            out.append(np.nan); continue
        y = np.zeros(e["n_atoms"], dtype=int); y[list(e["tox"])] = 1
        out.append(roc_auc_score(y, np.asarray(sc, dtype=float))
                   if len(np.unique(y)) > 1 else np.nan)
    return np.array(out)

rows = []
for label, sc in SCORES.items():
    a = attribution_auroc_per_molecule(sc)
    a = a[~np.isnan(a)]
    stat, p = wilcoxon(a - 0.5)
    from scipy.stats import rankdata
    d = a - 0.5; nz = d[d != 0]; rk = rankdata(np.abs(nz))
    rb = (rk[nz > 0].sum() - rk[nz < 0].sum()) / (rk[nz > 0].sum() + rk[nz < 0].sum())
    rows.append({"explainer": label, "n": len(a), "mean_auroc": a.mean(),
                 "median_auroc": float(np.median(a)),
                 "pct_above_half": 100 * (a > 0.5).mean(),
                 "wilcoxon_p_raw": p, "rank_biserial": rb})

sig = pd.DataFrame(rows)
p = sig["wilcoxon_p_raw"].values
order = np.argsort(p); k = len(p); adj = np.empty(k); run = 0.0
for r, i in enumerate(order):
    run = max(run, (k - r) * p[i]); adj[i] = min(run, 1.0)
sig["wilcoxon_p_holm"] = adj
sig["differs_from_uninformative"] = sig["wilcoxon_p_holm"] < 0.05
sig.to_csv(f"{OUTDIR}/table19_attribution_significance.csv", index=False)
print(sig.round(4).to_string(index=False))

                 explainer   n  mean_auroc  median_auroc  pct_above_half  wilcoxon_p_raw  rank_biserial  wilcoxon_p_holm  differs_from_uninformative
XGBoost (TreeSHAP, signed) 237      0.7979        0.8631         87.3418          0.0000         0.9036           0.0000                        True
GIN (Integrated Gradients) 237      0.6150        0.6250         70.4641          0.0000         0.4895           0.0000                        True
    GIN (Gradient x Input) 237      0.6051        0.6299         64.9789          0.0000         0.4700           0.0000                        True
            GIN (Saliency) 237      0.6132        0.6522         66.6667          0.0000         0.4846           0.0000                        True
        GIN (GNNExplainer) 237      0.4763        0.4481         41.3502          0.0319        -0.1622           0.0319                        True


## 6. Interval estimates for accuracy and calibration

Point estimates on a single test set carry sampling uncertainty. Bootstrap intervals are
computed for each model's ROC-AUC, for the pairwise differences between models, and for the
difference in expected calibration error between the two families. Resampling is paired: the
same resampled compounds are used for every model in a replicate.

In [11]:
# Calibration error conditioned on the true class. Scaffold partitioning shifts the
# positive rate between calibration and test sets, so a model may be miscalibrated to
# a different degree for actives than for inactives even when the pooled error is small.
def ece_equal_width_local(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e

zc = np.load(f"{OUTDIR}/predictions_primary.npz")
yc = zc["y_test"]
rows = []
for m in ["XGBoost", "RandomForest", "GCN", "GIN", "GAT"]:
    pc = zc[f"test_{m}"]
    pos, neg = yc == 1, yc == 0
    rows.append({"model": m,
                 "ECE_pooled": ece_equal_width_local(yc, pc),
                 "mean_probability_actives": float(pc[pos].mean()),
                 "mean_probability_inactives": float(pc[neg].mean()),
                 "conditional_error_actives": float(1.0 - pc[pos].mean()),
                 "conditional_error_inactives": float(pc[neg].mean())})
cw = pd.DataFrame(rows)
cw.to_csv(f"{OUTDIR}/table26_conditional_calibration.csv", index=False)
print(cw.round(4).to_string(index=False))

       model  ECE_pooled  mean_probability_actives  mean_probability_inactives  conditional_error_actives  conditional_error_inactives
     XGBoost      0.0586                    0.6346                      0.3993                     0.3654                       0.3993
RandomForest      0.0906                    0.6555                      0.4333                     0.3445                       0.4333
         GCN      0.0750                    0.6572                      0.3413                     0.3428                       0.3413
         GIN      0.0821                    0.7029                      0.3364                     0.2971                       0.3364
         GAT      0.0996                    0.6914                      0.3219                     0.3086                       0.3219


In [12]:
z = np.load(f"{OUTDIR}/predictions_primary.npz")
M = ["XGBoost", "RandomForest", "GCN", "GIN", "GAT"]
P  = {m: z[f"test_{m}"] for m in M}
yt = z["y_test"]

def ece_equal_width(y, p, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, n_bins - 1)
    e = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e

rng = np.random.default_rng(SEED)
idx = rng.integers(0, len(yt), (N_BOOTSTRAP, len(yt)))
boot_auc = {m: np.empty(N_BOOTSTRAP) for m in M}
boot_ece = {m: np.empty(N_BOOTSTRAP) for m in M}
for b in range(N_BOOTSTRAP):
    i = idx[b]; yb = yt[i]
    if len(np.unique(yb)) < 2:
        for m in M:
            boot_auc[m][b] = np.nan; boot_ece[m][b] = np.nan
        continue
    for m in M:
        boot_auc[m][b] = roc_auc_score(yb, P[m][i])
        boot_ece[m][b] = ece_equal_width(yb, P[m][i])

rows = []
for m in M:
    a = boot_auc[m]
    rows.append({"model": m, "ROC_AUC": roc_auc_score(yt, P[m]),
                 "ci_low": np.nanpercentile(a, 2.5),
                 "ci_high": np.nanpercentile(a, 97.5)})
auc_ci = pd.DataFrame(rows)
auc_ci.to_csv(f"{OUTDIR}/table20_auc_bootstrap_ci.csv", index=False)
print(auc_ci.round(4).to_string(index=False))

rows = []
for a in range(len(M)):
    for b in range(a + 1, len(M)):
        d = boot_auc[M[a]] - boot_auc[M[b]]
        d = d[~np.isnan(d)]
        rows.append({"model_a": M[a], "model_b": M[b],
                     "difference": roc_auc_score(yt, P[M[a]]) - roc_auc_score(yt, P[M[b]]),
                     "ci_low": np.percentile(d, 2.5), "ci_high": np.percentile(d, 97.5),
                     "p_two_sided": 2 * min((d > 0).mean(), (d < 0).mean())})
pair = pd.DataFrame(rows)
pair.to_csv(f"{OUTDIR}/table21_paired_bootstrap.csv", index=False)
print()
print(pair.round(4).to_string(index=False))

CL, GR = ["XGBoost", "RandomForest"], ["GCN", "GIN", "GAT"]
d_auc = (np.nanmean([boot_auc[m] for m in GR], 0) -
         np.nanmean([boot_auc[m] for m in CL], 0))
d_ece = (np.nanmean([boot_ece[m] for m in GR], 0) -
         np.nanmean([boot_ece[m] for m in CL], 0))
fam = []
for name, d in [("ROC_AUC graph - classical", d_auc), ("ECE graph - classical", d_ece)]:
    d = d[~np.isnan(d)]
    fam.append({"quantity": name, "mean": d.mean(),
                "ci_low": np.percentile(d, 2.5), "ci_high": np.percentile(d, 97.5),
                "p_two_sided": 2 * min((d > 0).mean(), (d < 0).mean())})
fam_df = pd.DataFrame(fam)
fam_df.to_csv(f"{OUTDIR}/table22_family_difference_ci.csv", index=False)
print()
print(fam_df.round(4).to_string(index=False))

       model  ROC_AUC  ci_low  ci_high
     XGBoost   0.7824  0.7484   0.8154
RandomForest   0.7994  0.7668   0.8307
         GCN   0.7877  0.7544   0.8198
         GIN   0.8133  0.7821   0.8435
         GAT   0.7977  0.7650   0.8299

     model_a      model_b  difference  ci_low  ci_high  p_two_sided
     XGBoost RandomForest     -0.0170 -0.0299  -0.0042       0.0110
     XGBoost          GCN     -0.0053 -0.0306   0.0196       0.6894
     XGBoost          GIN     -0.0309 -0.0542  -0.0077       0.0084
     XGBoost          GAT     -0.0154 -0.0406   0.0099       0.2320
RandomForest          GCN      0.0117 -0.0102   0.0340       0.2932
RandomForest          GIN     -0.0139 -0.0345   0.0067       0.1986
RandomForest          GAT      0.0017 -0.0198   0.0234       0.8548
         GCN          GIN     -0.0256 -0.0414  -0.0099       0.0030
         GCN          GAT     -0.0100 -0.0272   0.0064       0.2340
         GIN          GAT      0.0156 -0.0023   0.0336       0.0866

                

## 7. Conformal prediction: marginal and class-conditional

Two inductive conformal predictors are compared. The marginal predictor pools the
calibration scores of both classes and applies one quantile; the class-conditional
(Mondrian) predictor calibrates each class separately. The distinction matters here because
scaffold partitioning shifts the positive rate between the calibration and test sets, and a
marginal predictor derives its guarantee from an exchangeability assumption that such a
shift violates. Coverage is reported with exact binomial intervals and the tests across
models are corrected for multiplicity.

In [13]:
ALPHA = 0.10

def marginal_conformal(p_valid, y_valid, p_test, alpha=ALPHA):
    cal = np.where(y_valid == 1, 1 - p_valid, p_valid)
    n = len(cal)
    q = np.quantile(cal, min(1.0, np.ceil((n + 1) * (1 - alpha)) / n), method="higher")
    inc_pos, inc_neg = (1 - p_test) <= q, p_test <= q
    return inc_pos, inc_neg

def mondrian_conformal(p_valid, y_valid, p_test, alpha=ALPHA):
    """Class-conditional inductive conformal predictor."""
    cal = {c: np.sort(np.where(c == 1, 1 - p_valid[y_valid == c], p_valid[y_valid == c]))
           for c in (0, 1)}
    inc = {}
    for c in (0, 1):
        s = 1 - p_test if c == 1 else p_test
        n = len(cal[c])
        pval = np.array([(np.sum(cal[c] >= si) + 1) / (n + 1) for si in s])
        inc[c] = pval > alpha
    return inc[1], inc[0]

def conformal_metrics(inc_pos, inc_neg, y, name, scheme):
    covered = np.where(y == 1, inc_pos, inc_neg)
    size = inc_pos.astype(int) + inc_neg.astype(int)
    single = size == 1
    single_pred = np.where(inc_pos & single, 1, 0)
    k, N = int(covered.sum()), len(covered)
    lo = stats.beta.ppf(0.025, k, N - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(0.975, k + 1, N - k) if k < N else 1.0
    return {"model": name, "scheme": scheme, "nominal_coverage": 1 - ALPHA,
            "coverage": covered.mean(), "n_test": N,
            "coverage_ci_low": lo, "coverage_ci_high": hi,
            "p_vs_nominal": stats.binomtest(k, N, 1 - ALPHA).pvalue,
            "avg_set_size": size.mean(), "singleton_rate": single.mean(),
            "uncertain_rate": (size == 2).mean(), "empty_rate": (size == 0).mean(),
            "singleton_accuracy": float((single_pred[single] == y[single]).mean())
                                  if single.sum() else np.nan}

zz = np.load(f"{OUTDIR}/predictions_primary.npz")
MODELS_ORDER = ["XGBoost", "RandomForest", "GCN", "GIN", "GAT"]
rows = []
for scheme, fn in [("marginal", marginal_conformal), ("mondrian", mondrian_conformal)]:
    for m in MODELS_ORDER:
        ip, ineg = fn(zz[f"valid_{m}"], zz["y_valid"], zz[f"test_{m}"])
        rows.append(conformal_metrics(ip, ineg, zz["y_test"], m, scheme))

conf = pd.DataFrame(rows)
for scheme in ("marginal", "mondrian"):
    mask = conf.scheme == scheme
    p = conf.loc[mask, "p_vs_nominal"].values
    order = np.argsort(p); k = len(p); adj = np.empty(k); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (k - r) * p[i]); adj[i] = min(run, 1.0)
    conf.loc[mask, "p_holm"] = adj
conf["undercoverage_after_correction"] = ((conf.p_holm < 0.05) &
                                          (conf.coverage < conf.nominal_coverage))
conf.to_csv(f"{OUTDIR}/table03_conformal.csv", index=False)
for scheme in ("marginal", "mondrian"):
    print(f"--- {scheme} ---")
    print(conf[conf.scheme == scheme][
        ["model", "coverage", "coverage_ci_low", "coverage_ci_high", "p_holm",
         "avg_set_size", "singleton_rate", "undercoverage_after_correction"]]
        .round(4).to_string(index=False))
    print()

--- marginal ---
       model  coverage  coverage_ci_low  coverage_ci_high  p_holm  avg_set_size  singleton_rate  undercoverage_after_correction
     XGBoost    0.8912           0.8662            0.9129  0.4212        1.4394          0.5606                           False
RandomForest    0.8857           0.8603            0.9079  0.3881        1.4063          0.5937                           False
         GCN    0.8595           0.8321            0.8840  0.0026        1.3196          0.6804                            True
         GIN    0.8760           0.8498            0.8991  0.1410        1.3361          0.6639                           False
         GAT    0.8760           0.8498            0.8991  0.1410        1.3485          0.6515                           False

--- mondrian ---
       model  coverage  coverage_ci_low  coverage_ci_high  p_holm  avg_set_size  singleton_rate  undercoverage_after_correction
     XGBoost    0.8691           0.8424            0.8928  0.0384    

## 8. Descriptor and linear baselines

The classical arm so far uses circular fingerprints only. Two further reference points are
added: a linear model on the same fingerprints, which bounds what a non-interacting model
recovers, and a gradient-boosted model on RDKit two-dimensional descriptors, which is a
different molecular representation rather than a different learner. The descriptor model
receives the same search budget as the other models, and is then evaluated across the same
fifteen scaffold partitions so that it enters the family comparison on equal terms.

In [15]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 15.9 MB/s eta 0:00:00


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

desc_names = [d[0] for d in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)
DESC_PATH = f"{OUTDIR}/rdkit2d_descriptors.npz"

if os.path.exists(DESC_PATH):
    z_d = np.load(DESC_PATH, allow_pickle=True)
    DESC, DESC_SMILES = z_d["desc"], list(z_d["smiles"])
    print("Descriptors loaded from cache")
else:
    t0 = time.time()
    rows = []
    for s in df["canonical"]:
        m = Chem.MolFromSmiles(s)
        rows.append(list(calc.CalcDescriptors(m)) if m else [0.0] * len(desc_names))
    DESC = np.nan_to_num(np.array(rows, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    DESC_SMILES = list(df["canonical"])
    np.savez(DESC_PATH, desc=DESC, smiles=np.array(DESC_SMILES, dtype=object))
    print(f"{DESC.shape[1]} descriptors computed for {DESC.shape[0]} molecules "
          f"({(time.time()-t0)/60:.1f} min)")

DESC_INDEX = {s: i for i, s in enumerate(DESC_SMILES)}
def desc_for(frame):
    return DESC[[DESC_INDEX[s] for s in frame["canonical"]]]

D_train, D_valid, D_test = desc_for(df_train), desc_for(df_valid), desc_for(df_test)

HPD_PATH = f"{OUTDIR}/hpo_descriptor.json"
if os.path.exists(HPD_PATH):
    HP_DESC = json.load(open(HPD_PATH))
    print("Descriptor-model configuration loaded from cache")
else:
    def obj_desc(trial):
        m = XGBClassifier(
            n_estimators=1500,
            max_depth=trial.suggest_int("max_depth", 3, 12),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 12),
            scale_pos_weight=spw, eval_metric="auc", early_stopping_rounds=50,
            tree_method="hist", random_state=SEED, n_jobs=-1)
        m.fit(D_train, y_train, eval_set=[(D_valid, y_valid)], verbose=False)
        return roc_auc_score(y_valid, m.predict_proba(D_valid)[:, 1])
    st = optuna.create_study(direction="maximize",
                             sampler=optuna.samplers.TPESampler(seed=SEED),
                             pruner=optuna.pruners.MedianPruner(n_warmup_steps=5))
    st.optimize(obj_desc, n_trials=100, show_progress_bar=True)
    HP_DESC = st.best_params
    json.dump(HP_DESC, open(HPD_PATH, "w"), indent=2)
    print(f"Descriptor model validation ROC-AUC = {st.best_value:.4f}")
    print(f"  {HP_DESC}")

def fit_descriptor_xgb(Dtr, ytr, Dva, yva, seed):
    m = XGBClassifier(n_estimators=1500, **HP_DESC,
                      scale_pos_weight=(ytr == 0).sum() / max((ytr == 1).sum(), 1),
                      eval_metric="auc", early_stopping_rounds=50, tree_method="hist",
                      random_state=seed, n_jobs=-1)
    m.fit(Dtr, ytr, eval_set=[(Dva, yva)], verbose=False)
    return m

p_desc = np.mean([fit_descriptor_xgb(D_train, y_train, D_valid, y_valid, s)
                  .predict_proba(D_test)[:, 1] for s in SEEDS], axis=0)

HPL_PATH = f"{OUTDIR}/hpo_logreg.json"
if os.path.exists(HPL_PATH):
    HP_LR = json.load(open(HPL_PATH))
else:
    def obj_lr(trial):
        m = make_pipeline(
            StandardScaler(with_mean=False),
            LogisticRegression(max_iter=3000, class_weight="balanced",
                               C=trial.suggest_float("C", 1e-3, 1e2, log=True),
                               solver="liblinear",
                               penalty=trial.suggest_categorical("penalty", ["l1", "l2"])))
        m.fit(X_train, y_train)
        return roc_auc_score(y_valid, m.predict_proba(X_valid)[:, 1])
    stl = optuna.create_study(direction="maximize",
                              sampler=optuna.samplers.TPESampler(seed=SEED))
    stl.optimize(obj_lr, n_trials=100, show_progress_bar=True)
    HP_LR = stl.best_params
    json.dump(HP_LR, open(HPL_PATH, "w"), indent=2)
    print(f"Logistic regression validation ROC-AUC = {stl.best_value:.4f}  {HP_LR}")

logreg = make_pipeline(StandardScaler(with_mean=False),
                       LogisticRegression(max_iter=3000, class_weight="balanced",
                                          solver="liblinear", **HP_LR))
logreg.fit(X_train, y_train)
p_lr = logreg.predict_proba(X_test)[:, 1]

def best_threshold(y, p):
    grid = np.linspace(0.05, 0.95, 91)
    return grid[int(np.argmax([balanced_accuracy_score(y, (p >= t).astype(int))
                               for t in grid]))]

rows = []
for name, p_va, p_te in [
        ("LogisticRegression (ECFP4)", logreg.predict_proba(X_valid)[:, 1], p_lr),
        ("XGBoost (RDKit-2D)",
         np.mean([fit_descriptor_xgb(D_train, y_train, D_valid, y_valid, s)
                  .predict_proba(D_valid)[:, 1] for s in SEEDS], axis=0), p_desc)]:
    thr = best_threshold(y_valid, p_va)
    yhat = (p_te >= thr).astype(int)
    rows.append({"model": name, "ROC_AUC": roc_auc_score(y_test, p_te),
                 "PR_AUC": average_precision_score(y_test, p_te),
                 "balanced_acc": balanced_accuracy_score(y_test, yhat),
                 "sensitivity": recall_score(y_test, yhat, pos_label=1),
                 "specificity": recall_score(y_test, yhat, pos_label=0),
                 "MCC": matthews_corrcoef(y_test, yhat)})
extra = pd.DataFrame(rows)
extra.to_csv(f"{OUTDIR}/table24_extra_baselines.csv", index=False)
print(extra.round(4).to_string(index=False))
np.savez(f"{OUTDIR}/predictions_extra.npz", p_logreg=p_lr, p_descriptor=p_desc)

217 descriptors computed for 7255 molecules (1.4 min)


  0%|          | 0/100 [00:00<?, ?it/s]

Descriptor model validation ROC-AUC = 0.8509
  {'max_depth': 11, 'learning_rate': 0.04698572793108446, 'subsample': 0.7067690121894205, 'colsample_bytree': 0.612877264688783, 'reg_lambda': 0.007919413614819864, 'reg_alpha': 0.035122441585667276, 'min_child_weight': 3}


  0%|          | 0/100 [00:00<?, ?it/s]

Logistic regression validation ROC-AUC = 0.8040  {'C': 0.0111676008464174, 'penalty': 'l1'}
                     model  ROC_AUC  PR_AUC  balanced_acc  sensitivity  specificity    MCC
LogisticRegression (ECFP4)   0.7611  0.7141        0.7074       0.6250       0.7897 0.4217
        XGBoost (RDKit-2D)   0.8055  0.7949        0.7459       0.6994       0.7923 0.4945


In [17]:
# The descriptor model across the same fifteen scaffold partitions.
DESC_RUNS = f"{OUTDIR}/benchmark_runs_descriptor.csv"
done_parts = set()
if os.path.exists(DESC_RUNS):
    done_parts = set(pd.read_csv(DESC_RUNS)["partition"])
rows = pd.read_csv(DESC_RUNS).to_dict("records") if done_parts else []

for sd in range(42, 57):
    part = f"scaffold_{sd}"
    if part in done_parts:
        continue
    d_tr, d_va, d_te = [x.reset_index(drop=True)
                        for x in make_scaffold_split(df, seed=sd)]
    Dtr, Dva, Dte = desc_for(d_tr), desc_for(d_va), desc_for(d_te)
    ytr, yva, yte = (d_tr["label"].values, d_va["label"].values, d_te["label"].values)
    pv = np.mean([fit_descriptor_xgb(Dtr, ytr, Dva, yva, s).predict_proba(Dva)[:, 1]
                  for s in SEEDS], axis=0)
    pt = np.mean([fit_descriptor_xgb(Dtr, ytr, Dva, yva, s).predict_proba(Dte)[:, 1]
                  for s in SEEDS], axis=0)
    thr = best_threshold(yva, pv); yhat = (pt >= thr).astype(int)
    rows.append({"partition": part, "scheme": "scaffold", "seed": sd,
                 "model": "XGBoost_RDKit2D", "arm": "descriptor",
                 "ROC_AUC": roc_auc_score(yte, pt),
                 "PR_AUC": average_precision_score(yte, pt),
                 "balanced_acc": balanced_accuracy_score(yte, yhat),
                 "sensitivity": recall_score(yte, yhat, pos_label=1),
                 "specificity": recall_score(yte, yhat, pos_label=0),
                 "MCC": matthews_corrcoef(yte, yhat)})
    pd.DataFrame(rows).to_csv(DESC_RUNS, index=False)
    print(f"  {part}  ROC-AUC {rows[-1]['ROC_AUC']:.4f}")

desc_runs = pd.read_csv(DESC_RUNS)
bench = pd.read_csv(f"{OUTDIR}/benchmark_runs.csv")
sc = bench[(bench.scheme == "scaffold") & (bench.arm == "main")]
piv = sc.pivot_table(index="partition", columns="model", values="ROC_AUC")
piv["XGBoost_RDKit2D"] = desc_runs.set_index("partition")["ROC_AUC"]
piv = piv.dropna()

from scipy.stats import friedmanchisquare, rankdata
cols = ["XGBoost", "RandomForest", "XGBoost_RDKit2D", "GCN", "GIN", "GAT"]
piv = piv[cols]
N, k = piv.shape
stat, p = friedmanchisquare(*[piv[c].values for c in cols])
Q = {5: 2.728, 6: 2.850, 7: 2.949}
CD = Q[k] * np.sqrt(k * (k + 1) / (6 * N))
ranks = pd.Series(np.mean([rankdata(-r) for r in piv.values], axis=0),
                  index=cols).sort_values()
print(f"\nSix models over {N} scaffold partitions: chi2 = {stat:.3f}, p = {p:.2e}")
print(f"Nemenyi critical difference: {CD:.3f}")
print(piv.mean().round(4).to_string())
print()
print(ranks.round(2).to_string())

d = piv["GIN"] - piv["XGBoost_RDKit2D"]
nz = d[d != 0]; rk = rankdata(np.abs(nz))
rb = (rk[nz > 0].sum() - rk[nz < 0].sum()) / (rk[nz > 0].sum() + rk[nz < 0].sum())
stat2, p2 = wilcoxon(d)
print(f"\nGIN - XGBoost(RDKit-2D): mean {d.mean():+.4f}, p = {p2:.4f}, "
      f"r_rb = {rb:+.3f}, GIN ahead on {int((d>0).sum())}/{len(d)} partitions")
piv.to_csv(f"{OUTDIR}/table25_six_model_benchmark.csv")
pd.DataFrame({"model": ranks.index, "mean_rank": ranks.values,
              "mean_ROC_AUC": [piv[m].mean() for m in ranks.index]}).to_csv(
    f"{OUTDIR}/table25b_six_model_ranks.csv", index=False)

  scaffold_42  ROC-AUC 0.7851
  scaffold_43  ROC-AUC 0.7972
  scaffold_44  ROC-AUC 0.8134
  scaffold_45  ROC-AUC 0.8123
  scaffold_46  ROC-AUC 0.8134
  scaffold_47  ROC-AUC 0.8100
  scaffold_48  ROC-AUC 0.8030
  scaffold_49  ROC-AUC 0.7971
  scaffold_50  ROC-AUC 0.8011
  scaffold_51  ROC-AUC 0.7915
  scaffold_52  ROC-AUC 0.8056
  scaffold_53  ROC-AUC 0.8144
  scaffold_54  ROC-AUC 0.7922
  scaffold_55  ROC-AUC 0.8037
  scaffold_56  ROC-AUC 0.7913

Six models over 15 scaffold partitions: chi2 = 42.314, p = 5.09e-08
Nemenyi critical difference: 1.947
model
XGBoost            0.7801
RandomForest       0.7949
XGBoost_RDKit2D    0.8021
GCN                0.7909
GIN                0.8024
GAT                0.7957

GIN                2.00
XGBoost_RDKit2D    2.07
GAT                3.27
RandomForest       3.60
GCN                4.40
XGBoost            5.67

GIN - XGBoost(RDKit-2D): mean +0.0003, p = 0.8904, r_rb = +0.050, GIN ahead on 8/15 partitions


In [18]:
# The descriptor model on the fifteen random partitions, so that it enters the
# partition-scheme comparison on the same footing as the other models.
RAND_RUNS = f"{OUTDIR}/benchmark_runs_descriptor_random.csv"
done_r = set(pd.read_csv(RAND_RUNS)["partition"]) if os.path.exists(RAND_RUNS) else set()
rrows = pd.read_csv(RAND_RUNS).to_dict("records") if done_r else []

def make_random_split_local(d, seed, frac=(0.8, 0.1, 0.1)):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(d.index.values)
    n = len(d); a, b = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    return d.loc[idx[:a]], d.loc[idx[a:b]], d.loc[idx[b:]]

for sd in range(42, 57):
    part = f"random_{sd}"
    if part in done_r:
        continue
    d_tr, d_va, d_te = [x.reset_index(drop=True)
                        for x in make_random_split_local(df, seed=sd)]
    Dtr, Dva, Dte = desc_for(d_tr), desc_for(d_va), desc_for(d_te)
    ytr, yva, yte = (d_tr["label"].values, d_va["label"].values, d_te["label"].values)
    pv = np.mean([fit_descriptor_xgb(Dtr, ytr, Dva, yva, s).predict_proba(Dva)[:, 1]
                  for s in SEEDS], axis=0)
    pt = np.mean([fit_descriptor_xgb(Dtr, ytr, Dva, yva, s).predict_proba(Dte)[:, 1]
                  for s in SEEDS], axis=0)
    thr = best_threshold(yva, pv); yhat = (pt >= thr).astype(int)
    rrows.append({"partition": part, "scheme": "random", "seed": sd,
                  "model": "XGBoost_RDKit2D", "arm": "descriptor",
                  "ROC_AUC": roc_auc_score(yte, pt),
                  "PR_AUC": average_precision_score(yte, pt),
                  "balanced_acc": balanced_accuracy_score(yte, yhat),
                  "MCC": matthews_corrcoef(yte, yhat)})
    pd.DataFrame(rrows).to_csv(RAND_RUNS, index=False)
    print(f"  {part}  ROC-AUC {rrows[-1]['ROC_AUC']:.4f}")

d_sc = pd.read_csv(DESC_RUNS)["ROC_AUC"].mean()
d_rn = pd.read_csv(RAND_RUNS)["ROC_AUC"].mean()
print(f"\nXGBoost (RDKit-2D): scaffold {d_sc:.4f}, random {d_rn:.4f}, "
      f"inflation {d_rn - d_sc:+.4f}")

# Calibration and conformal for the descriptor model on the primary partition.
def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1 - eps); return np.log(p / (1 - p))

def fit_temperature(logits, y):
    zt = torch.tensor(logits, dtype=torch.float32)
    tt = torch.tensor(y, dtype=torch.float32)
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=200)
    lossf = torch.nn.BCEWithLogitsLoss()
    def closure():
        opt.zero_grad(); l = lossf(zt / torch.exp(log_T), tt); l.backward(); return l
    opt.step(closure)
    return float(torch.exp(log_T).item())

p_desc_valid = np.mean([fit_descriptor_xgb(D_train, y_train, D_valid, y_valid, s)
                        .predict_proba(D_valid)[:, 1] for s in SEEDS], axis=0)
T_desc = fit_temperature(to_logit(p_desc_valid), y_valid)
p_desc_cal = 1 / (1 + np.exp(-to_logit(p_desc) / T_desc))
desc_cal = {"model": "XGBoost (RDKit-2D)", "temperature": T_desc,
            "ECE_raw": ece_equal_width_local(y_test, p_desc),
            "ECE_calibrated": ece_equal_width_local(y_test, p_desc_cal)}
for scheme, fn in [("marginal", marginal_conformal), ("mondrian", mondrian_conformal)]:
    ip, ineg = fn(p_desc_valid, y_valid, p_desc)
    m = conformal_metrics(ip, ineg, y_test, "XGBoost (RDKit-2D)", scheme)
    desc_cal[f"coverage_{scheme}"] = m["coverage"]
    desc_cal[f"avg_set_size_{scheme}"] = m["avg_set_size"]
pd.DataFrame([desc_cal]).to_csv(f"{OUTDIR}/table27_descriptor_calibration.csv", index=False)
for k, v in desc_cal.items():
    print(f"  {k:26s}: {v if isinstance(v, str) else round(float(v), 4)}")

  random_42  ROC-AUC 0.9264
  random_43  ROC-AUC 0.9162
  random_44  ROC-AUC 0.9229
  random_45  ROC-AUC 0.9192
  random_46  ROC-AUC 0.9022
  random_47  ROC-AUC 0.9196
  random_48  ROC-AUC 0.9223
  random_49  ROC-AUC 0.9254
  random_50  ROC-AUC 0.9121
  random_51  ROC-AUC 0.9122
  random_52  ROC-AUC 0.9305
  random_53  ROC-AUC 0.9203
  random_54  ROC-AUC 0.9135
  random_55  ROC-AUC 0.9251
  random_56  ROC-AUC 0.9439

XGBoost (RDKit-2D): scaffold 0.8021, random 0.9208, inflation +0.1187
  model                     : XGBoost (RDKit-2D)
  temperature               : 0.973
  ECE_raw                   : 0.0978
  ECE_calibrated            : 0.1009
  coverage_marginal         : 0.8774
  avg_set_size_marginal     : 1.3843
  coverage_mondrian         : 0.8815
  avg_set_size_mondrian     : 1.3664


## 9. Attribution analysis on further partitions

The comparison between trained and untrained attributions is carried out on the primary
partition. Repeating it on two further scaffold partitions establishes whether the
proportion of the above-chance signal that exists before training is a property of the
architecture and the attribution method, or of one particular partition. A reduced number of
draws is used, sufficient to place the proportion within an interval rather than to
estimate it as precisely as on the primary partition.

In [19]:
N_UNTRAINED_REP, N_TRAINED_REP = 100, 10
REPLICATE_SEEDS = [201, 202]
REP_PATH = f"{OUTDIR}/table28_attribution_replication.csv"
rep_rows = pd.read_csv(REP_PATH).to_dict("records") if os.path.exists(REP_PATH) else []
done_rep = {r["partition"] for r in rep_rows}

def eval_set_for(d_te, y_te):
    ev = []
    for smi, lab in zip(d_te["canonical"], y_te):
        if lab != 1:
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or mol.GetNumAtoms() == 0:
            continue
        tox = toxicophore_atoms(mol)
        if not tox or len(tox) == mol.GetNumAtoms():
            continue
        ev.append({"smiles": smi, "tox": tox, "n_atoms": mol.GetNumAtoms(),
                   "chance": len(tox) / mol.GetNumAtoms()})
    return ev

def overlap_generic(ev, scores_list, k_frac=K_FRAC, tie_seed=0):
    rng = np.random.default_rng(tie_seed)
    out = np.full(len(ev), np.nan)
    for j, (e, sc) in enumerate(zip(ev, scores_list)):
        if sc is None or len(sc) != e["n_atoms"]:
            continue
        n = e["n_atoms"]; k = max(1, int(round(k_frac * n)))
        order = np.lexsort((rng.random(n), -np.asarray(sc, dtype=float)))
        out[j] = len(set(order[:k].tolist()) & e["tox"]) / k
    return out

def lift_generic(ev, ov):
    ch = np.array([e["chance"] for e in ev]); m = ~np.isnan(ov)
    return float(ov[m].mean() / ch[m].mean()) if m.sum() else np.nan

for sd in REPLICATE_SEEDS:
    part = f"scaffold_{sd}"
    if part in done_rep:
        print(f"{part}: loaded from checkpoint"); continue
    t0 = time.time()
    d_tr, d_va, d_te = [x.reset_index(drop=True)
                        for x in make_scaffold_split(df, seed=sd)]
    y_te_r = d_te["label"].values
    tr_r, va_r = build_dataset(d_tr), build_dataset(d_va)
    ev = eval_set_for(d_te, y_te_r)
    graphs = [smiles_to_data(e["smiles"], 1.0) for e in ev]
    yv = d_tr["label"].values
    pw = torch.tensor([(yv == 0).sum() / max((yv == 1).sum(), 1)],
                      dtype=torch.float, device=DEVICE)

    null_lifts = []
    for j in range(N_UNTRAINED_REP):
        set_seed(30_000 + j)
        m = build_gnn("gin", hp["hidden"], hp["n_layers"], hp["dropout"],
                      hp.get("residual", True))
        sc = batched_gradient_scores(m, graphs, "ig")
        null_lifts.append(lift_generic(ev, overlap_generic(ev, sc, tie_seed=30_000 + j)))
        del m; torch.cuda.empty_cache()

    tr_lifts = []
    for j in range(N_TRAINED_REP):
        m, _ = train_gnn("gin", HP_GNN["gin"], 40_000 + j, tr_r, va_r, pw, 250, 30)
        sc = batched_gradient_scores(m, graphs, "ig")
        tr_lifts.append(lift_generic(ev, overlap_generic(ev, sc, tie_seed=40_000 + j)))
        del m; torch.cuda.empty_cache()

    nl, tl = np.array(null_lifts), np.array(tr_lifts)
    rep_rows.append({"partition": part, "n_molecules": len(ev),
                     "n_untrained": len(nl), "n_trained": len(tl),
                     "untrained_lift_mean": nl.mean(), "untrained_lift_sd": nl.std(ddof=1),
                     "untrained_p2_5": np.percentile(nl, 2.5),
                     "untrained_p97_5": np.percentile(nl, 97.5),
                     "trained_lift_mean": tl.mean(), "trained_lift_sd": tl.std(ddof=1),
                     "pretraining_fraction": (nl.mean() - 1) / (tl.mean() - 1),
                     "minutes": (time.time() - t0) / 60})
    pd.DataFrame(rep_rows).to_csv(REP_PATH, index=False)
    print(f"{part}: untrained {nl.mean():.4f}, trained {tl.mean():.4f}, "
          f"pre-training fraction {rep_rows[-1]['pretraining_fraction']:.3f}  "
          f"({rep_rows[-1]['minutes']:.1f} min)")

rep = pd.read_csv(REP_PATH)
main = pd.read_csv(f"{OUTDIR}/table13_training_gain.csv").iloc[0]
print(f"\nprimary (scaffold_{PRIMARY_SPLIT_SEED}): "
      f"pre-training fraction {main['pretraining_fraction']:.3f} "
      f"[{main['pretraining_frac_ci_low']:.3f}, {main['pretraining_frac_ci_high']:.3f}]")
print(rep[["partition", "n_molecules", "untrained_lift_mean",
           "trained_lift_mean", "pretraining_fraction"]].round(4).to_string(index=False))

scaffold_201: untrained 1.1266, trained 1.1880, pre-training fraction 0.674  (24.1 min)
scaffold_202: untrained 1.1404, trained 1.1514, pre-training fraction 0.927  (27.0 min)

primary (scaffold_101): pre-training fraction 0.478 [0.425, 0.543]
   partition  n_molecules  untrained_lift_mean  trained_lift_mean  pretraining_fraction
scaffold_201          243               1.1266             1.1880                0.6735
scaffold_202          226               1.1404             1.1514                0.9269


## 10. Per-seed variation

Reported predictions are the mean of three seeds. The individual seeds are evaluated so
that the spread hidden by the ensemble is on record.

In [20]:
rows = []
spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
for s in SEEDS:
    m = XGBClassifier(n_estimators=1500, **HP_CLASSIC["XGBoost"], scale_pos_weight=spw,
                      eval_metric="auc", early_stopping_rounds=50, tree_method="hist",
                      random_state=s, n_jobs=-1)
    m.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    rows.append({"model": "XGBoost", "seed": s,
                 "ROC_AUC": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])})
    m = RandomForestClassifier(**HP_CLASSIC["RandomForest"], class_weight="balanced",
                               random_state=s, n_jobs=-1).fit(X_train, y_train)
    rows.append({"model": "RandomForest", "seed": s,
                 "ROC_AUC": roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])})

for kind in ["gcn", "gin", "gat"]:
    hpk = HP_GNN[kind]
    for s in SEEDS:
        path = f"{MODELS}/{kind}_seed{s}.pt"
        if not os.path.exists(path):
            print(f"  {path} missing; skipped")
            continue
        m = build_gnn(kind, hpk["hidden"], hpk["n_layers"], hpk["dropout"],
                      hpk.get("residual", True))
        m.load_state_dict(torch.load(path, map_location=DEVICE)["state_dict"])
        _, p_te, _ = evaluate(m, test_loader)
        rows.append({"model": kind.upper(), "seed": s,
                     "ROC_AUC": roc_auc_score(y_test, p_te)})
        del m; torch.cuda.empty_cache()

seed_df = pd.DataFrame(rows)
seed_df.to_csv(f"{OUTDIR}/table23_per_seed.csv", index=False)
print(seed_df.pivot_table(index="model", columns="seed",
                          values="ROC_AUC").round(4).to_string())
print()
print(seed_df.groupby("model")["ROC_AUC"].agg(["mean", "std", "min", "max"])
            .round(4).to_string())

seed              42      43      44
model                               
GAT           0.7811  0.7763  0.8000
GCN           0.7774  0.7880  0.7556
GIN           0.7997  0.7960  0.7977
RandomForest  0.7999  0.7972  0.8005
XGBoost       0.7833  0.7626  0.7769

                mean     std     min     max
model                                       
GAT           0.7858  0.0125  0.7763  0.8000
GCN           0.7737  0.0165  0.7556  0.7880
GIN           0.7978  0.0019  0.7960  0.7997
RandomForest  0.7992  0.0017  0.7972  0.8005
XGBoost       0.7743  0.0106  0.7626  0.7833


## 11. Data randomisation with regularisation disabled

A model trained on permuted labels is only an informative control if it has fitted those
labels. Dropout, weight decay and class weighting are therefore disabled and training runs
to a fixed budget with no early stopping. The training AUC on the permuted labels records
how far each model fitted the randomised target, and training is repeated with a longer
budget if the first attempt leaves the network close to a constant output.

In [21]:
rng = np.random.default_rng(SEED)
y_orig = np.array([float(g.y.item()) for g in train_set])
y_perm = rng.permutation(y_orig)
print(f"Permutation check: agreement with the original labels = "
      f"{(y_orig == y_perm).mean():.3f}")

train_perm = []
for g, yp in zip(train_set, y_perm):
    h = g.clone(); h.y = torch.tensor([yp], dtype=torch.float); train_perm.append(h)
perm_loader = DataLoader(train_perm, batch_size=256)

def fit_permuted_gin(n_epochs, seed):
    m, loss = train_to_fit("gin", HP_GNN["gin"], seed, train_perm, n_epochs,
                           dropout=0.0, weight_decay=0.0)
    y_tr, p_tr, _ = evaluate(m, perm_loader)
    return m, loss, roc_auc_score(y_tr, p_tr), float(p_tr.std())

print("\nFitting the permuted-label GIN")
t0 = time.time()
m_perm, loss, auc_fit, sd_fit = fit_permuted_gin(300, SEED)
print(f"  training AUC on permuted labels = {auc_fit:.3f}, "
      f"prediction sd = {sd_fit:.4f}, loss = {loss:.4f}")
if auc_fit < 0.70 or sd_fit < 1e-3:
    print("  insufficient fit; repeating with a longer budget")
    del m_perm; torch.cuda.empty_cache()
    m_perm, loss, auc_fit, sd_fit = fit_permuted_gin(600, SEED + 1)
    print(f"  training AUC on permuted labels = {auc_fit:.3f}, "
          f"prediction sd = {sd_fit:.4f}, loss = {loss:.4f}")
print(f"  elapsed {(time.time()-t0)/60:.1f} min")

_, p_te, _ = evaluate(m_perm, test_loader)
ov = overlap_from_scores(batched_gradient_scores(m_perm, EVAL_GRAPHS, "ig"), tie_seed=SEED)
graph_row = {"arm": "graph", "model": "GIN (permuted labels)",
             "train_auc_permuted": auc_fit, "test_auc_true": roc_auc_score(y_test, p_te),
             "pred_sd": float(p_te.std()), "final_train_loss": loss,
             "lift": lift_from_overlap(ov), "n": int((~np.isnan(ov)).sum())}
del m_perm; torch.cuda.empty_cache()

y_train_perm = rng.permutation(y_train)
xgb_cfg = {k: v for k, v in HP_CLASSIC["XGBoost"].items()
           if k not in ("reg_alpha", "reg_lambda")}
xgb_perm = XGBClassifier(n_estimators=1500, **xgb_cfg, reg_alpha=0.0, reg_lambda=0.0,
                         eval_metric="auc", tree_method="hist",
                         random_state=SEED, n_jobs=-1)
xgb_perm.fit(X_train, y_train_perm, verbose=False)
p_tr_x = xgb_perm.predict_proba(X_train)[:, 1]
p_te_x = xgb_perm.predict_proba(X_test)[:, 1]
ov = overlap_from_scores(shap_to_atom_scores(tree_shap_matrix(xgb_perm, X_eval)),
                         tie_seed=SEED)
tree_row = {"arm": "tree", "model": "XGBoost (permuted labels)",
            "train_auc_permuted": roc_auc_score(y_train_perm, p_tr_x),
            "test_auc_true": roc_auc_score(y_test, p_te_x),
            "pred_sd": float(p_te_x.std()), "final_train_loss": np.nan,
            "lift": lift_from_overlap(ov), "n": int((~np.isnan(ov)).sum())}

drt = pd.DataFrame([graph_row, tree_row])
drt.to_csv(f"{OUTDIR}/table14_label_permuted.csv", index=False)
print()
print(drt.round(4).to_string(index=False))

Permutation check: agreement with the original labels = 0.514

Fitting the permuted-label GIN
    epoch  100  training loss = 0.6870
    epoch  200  training loss = 0.6870
    epoch  300  training loss = 0.6870
  training AUC on permuted labels = 0.500, prediction sd = 0.0000, loss = 0.6870
  insufficient fit; repeating with a longer budget
    epoch  100  training loss = 0.6855
    epoch  200  training loss = 0.6752
    epoch  300  training loss = 0.6637
    epoch  400  training loss = 0.6400
    epoch  500  training loss = 0.6180
    epoch  600  training loss = 0.5972
  training AUC on permuted labels = 0.705, prediction sd = 0.1886, loss = 0.5972
  elapsed 25.3 min

  arm                     model  train_auc_permuted  test_auc_true  pred_sd  final_train_loss   lift   n
graph     GIN (permuted labels)              0.7051         0.5652   0.1659            0.5972 1.1495 237
 tree XGBoost (permuted labels)              0.9727         0.5160   0.3126               NaN 0.9606 237


## 12. Summary

In [22]:
print("Artefacts written to", OUTDIR)
for f in ["table03_conformal.csv", "table14_label_permuted.csv",
          "table17_topk_sensitivity.csv", "table18_bit_collisions.csv",
          "table19_attribution_significance.csv", "table19b_lift_significance.csv",
          "table20_auc_bootstrap_ci.csv", "table21_paired_bootstrap.csv",
          "table22_family_difference_ci.csv", "table23_per_seed.csv",
          "table24_extra_baselines.csv", "table25_six_model_benchmark.csv",
          "table25b_six_model_ranks.csv", "table26_conditional_calibration.csv",
          "table27_descriptor_calibration.csv",
          "table28_attribution_replication.csv",
          "benchmark_runs_descriptor.csv", "benchmark_runs_descriptor_random.csv",
          "hpo_descriptor.json", "hpo_logreg.json", "scaffold_leakage.csv"]:
    print("  -", f)

Artefacts written to /content/drive/MyDrive/ames_pipeline/results
  - table03_conformal.csv
  - table14_label_permuted.csv
  - table17_topk_sensitivity.csv
  - table18_bit_collisions.csv
  - table19_attribution_significance.csv
  - table19b_lift_significance.csv
  - table20_auc_bootstrap_ci.csv
  - table21_paired_bootstrap.csv
  - table22_family_difference_ci.csv
  - table23_per_seed.csv
  - table24_extra_baselines.csv
  - table25_six_model_benchmark.csv
  - table25b_six_model_ranks.csv
  - table26_conditional_calibration.csv
  - table27_descriptor_calibration.csv
  - table28_attribution_replication.csv
  - benchmark_runs_descriptor.csv
  - benchmark_runs_descriptor_random.csv
  - hpo_descriptor.json
  - hpo_logreg.json
  - scaffold_leakage.csv


In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!cd /content/drive/MyDrive/ames_pipeline && zip -qr /content/part5_results.zip results
from google.colab import files
files.download("/content/part5_results.zip")

Mounted at /content/drive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>